In [ ]:
!pip install datasets

In [ ]:
import numpy as np
import pandas as pd
import json
import re


Data Exploration


In [ ]:
from datasets import load_dataset

In [ ]:
dataset=load_dataset("lavita/MedQuAD")

README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001-e36383d177026d(…): reconstructing file:   0%|          |  0.00B / 10.7MB            

data/train-00000-of-00001-e36383d177026d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
        num_rows: 47441
    })
})


In [ ]:
dataset.keys()

dict_keys(['train'])

In [ ]:
train_data=dataset['train']

In [ ]:
len(train_data)

47441

In [ ]:
train_data.column_names

['document_id',
 'document_source',
 'document_url',
 'category',
 'umls_cui',
 'umls_semantic_types',
 'umls_semantic_group',
 'synonyms',
 'question_id',
 'question_focus',
 'question_type',
 'question',
 'answer']

In [ ]:
train_data.features

{'document_id': Value('string'),
 'document_source': Value('string'),
 'document_url': Value('string'),
 'category': Value('string'),
 'umls_cui': Value('string'),
 'umls_semantic_types': Value('string'),
 'umls_semantic_group': Value('string'),
 'synonyms': Value('string'),
 'question_id': Value('string'),
 'question_focus': Value('string'),
 'question_type': Value('string'),
 'question': Value('string'),
 'answer': Value('string')}

In [ ]:
train_data[0]

{'document_id': '0000559',
 'document_source': 'GHR',
 'document_url': 'https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair',
 'category': None,
 'umls_cui': 'C0343073',
 'umls_semantic_types': 'T047',
 'umls_semantic_group': 'Disorders',
 'synonyms': 'KWWH',
 'question_id': '0000559-1',
 'question_focus': 'keratoderma with woolly hair',
 'question_type': 'information',
 'question': 'What is (are) keratoderma with woolly hair ?',
 'answer': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the

In [ ]:
for i in range(5):
  print(train_data[i])

{'document_id': '0000559', 'document_source': 'GHR', 'document_url': 'https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair', 'category': None, 'umls_cui': 'C0343073', 'umls_semantic_types': 'T047', 'umls_semantic_group': 'Disorders', 'synonyms': 'KWWH', 'question_id': '0000559-1', 'question_focus': 'keratoderma with woolly hair', 'question_type': 'information', 'question': 'What is (are) keratoderma with woolly hair ?', 'answer': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of th

In [ ]:
df=train_data.to_pandas()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47441 entries, 0 to 47440
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   document_id          47436 non-null  object
 1   document_source      47441 non-null  object
 2   document_url         47441 non-null  object
 3   category             32010 non-null  object
 4   umls_cui             31417 non-null  object
 5   umls_semantic_types  31375 non-null  object
 6   umls_semantic_group  31417 non-null  object
 7   synonyms             24669 non-null  object
 8   question_id          47441 non-null  object
 9   question_focus       47427 non-null  object
 10  question_type        47441 non-null  object
 11  question             47441 non-null  object
 12  answer               16407 non-null  object
dtypes: object(13)
memory usage: 4.7+ MB


In [ ]:
df.shape

(47441, 13)

In [ ]:
df.isnull().sum()

,0
document_id,5
document_source,0
document_url,0
category,15431
umls_cui,16024
umls_semantic_types,16066
umls_semantic_group,16024
synonyms,22772
question_id,0
question_focus,14


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df['question'].str.len().describe()

,question
count,47441.000000
mean,51.537531
std,15.771672
min,14.000000
25%,40.000000
50%,51.000000
75%,62.000000
max,191.000000


In [ ]:
df['answer'].str.len().describe()

,answer
count,16407.000000
mean,1303.452673
std,1656.694326
min,6.000000
25%,487.000000
50%,890.000000
75%,1589.000000
max,29046.000000


In [ ]:
df.to_csv("medquad_raw.csv",index=False)

Data Preprocessing


In [ ]:
df=pd.read_csv("medquad_raw.csv")

In [ ]:
df.columns.tolist()

['document_id',
 'document_source',
 'document_url',
 'category',
 'umls_cui',
 'umls_semantic_types',
 'umls_semantic_group',
 'synonyms',
 'question_id',
 'question_focus',
 'question_type',
 'question',
 'answer']

In [ ]:
# remove missing question/answer
df = df.dropna(subset=["question", "answer"])

print("Shape after removing missing values:", df.shape)

Shape after removing missing values: (16407, 13)


In [ ]:
# clean whitespace
df["question"] = df["question"].str.strip()
df["answer"] = df["answer"].str.strip()

In [ ]:
before = len(df)

df = df.drop_duplicates(
    subset=["question", "answer"]
).reset_index(drop=True)

after = len(df)

print("Duplicates removed:", before - after)
print("Final records:", after)

Duplicates removed: 48
Final records: 16359


In [ ]:
print(df.head(2))

  document_id document_source  \
0     0000559             GHR   
1     0000559             GHR   
2     0000559             GHR   
3     0000559             GHR   
4     0000559             GHR   

                                        document_url category  umls_cui  \
0  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
1  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
2  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
3  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
4  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   

  umls_semantic_types umls_semantic_group synonyms question_id  \
0                T047           Disorders     KWWH   0000559-1   
1                T047           Disorders     KWWH   0000559-2   
2                T047           Disorders     KWWH   0000559-3   
3                T047           Disorders     KWWH   0000559-4   
4                T04

In [ ]:
def create_instruction(row):
  return {
       "instruction": "Answer the following medical question accurately and clearly.",
        "input": row["question"],
        "output": row["answer"]
  }

In [ ]:
processed_data=df.apply(create_instruction,axis=1).tolist()

In [ ]:
print(processed_data[0])

{'instruction': 'Answer the following medical question accurately and clearly.', 'input': 'What is (are) keratoderma with woolly hair ?', 'output': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appe

In [ ]:
def create_training_text(row):
    return (
        "### Instruction:\n"
        "Answer the following medical question accurately and clearly.\n\n"
        "### Question:\n"
        f"{row['question']}\n\n"
        "### Answer:\n"
        f"{row['answer']}"
    )

In [ ]:
df["text"]=df.apply(create_training_text,axis=1)

In [ ]:
print(df["text"][0])

### Instruction:
Answer the following medical question accurately and clearly.

### Question:
What is (are) keratoderma with woolly hair ?

### Answer:
Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not a

In [ ]:
print("Final dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Final dataset shape: (16359, 14)

Columns:
['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer', 'text']


In [ ]:
df[['question','answer','text']].to_csv("processed.csv",index=False)

In [ ]:
with open(
    "medical_training.jslon",
    "w",
    encoding="utf-8"
)as file:
  for item in processed_data:
    file.write(json.dumps(item,ensure_ascii=False)+"\n")


In [ ]:
with open(
    "medical_training.jslon",
    "r",
    encoding="utf-8"
) as f:
    first_line = f.readline()

print(first_line)

{"instruction": "Answer the following medical question accurately and clearly.", "input": "What is (are) keratoderma with woolly hair ?", "output": "Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appe

In [ ]:
print("Final number of records:", len(df))

print("\nMissing values:")
print(df[["question", "answer", "text"]].isnull().sum())

print("\nDuplicate records:")
print(df[["question", "answer"]].duplicated().sum())

Final number of records: 16359

Missing values:
question    0
answer      0
text        0
dtype: int64

Duplicate records:
0


Base Model Selection & Loading

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")


PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Model:", MODEL_NAME)

Model: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
print("tokenizer loaded successfully")

tokenizer loaded successfully


In [ ]:
text="what is diabates"
tokens=tokenizer(text)
print(tokens)


{'input_ids': [12555, 374, 1853, 370, 973], 'attention_mask': [1, 1, 1, 1, 1]}


In [ ]:
print("Input_ids")
print(tokens['input_ids'])

Input_ids
[12555, 374, 1853, 370, 973]


In [ ]:
model=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16

)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("Model loaded on:", device)

Model loaded on: cuda


In [ ]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print(
    "Model type:",
    model.config.model_type
)

print(
    "Hidden size:",
    model.config.hidden_size
)

print(
    "Number of layers:",
    model.config.num_hidden_layers
)

print(
    "Vocabulary size:",
    model.config.vocab_size
)
if torch.cuda.is_available():

    allocated = (
        torch.cuda.memory_allocated() / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved() / 1024**3
    )

    print("\nGPU Memory")
    print(
        f"Allocated: {allocated:.2f} GB"
    )

    print(
        f"Reserved: {reserved:.2f} GB"
    )

Total parameters: 1,543,714,304
Model type: qwen2
Hidden size: 1536
Number of layers: 28
Vocabulary size: 151936

GPU Memory
Allocated: 2.88 GB
Reserved: 3.06 GB


In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is diabetes?"
    }
]

# Convert conversation into model's chat format
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\n" + "=" * 60)
print("BASE MODEL TEST")
print("=" * 60)

# Tokenize prompt
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

# Generate response
with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True
    )

# Remove original prompt tokens
input_length = inputs["input_ids"].shape[1]

generated_tokens = outputs[0][input_length:]

# Convert tokens back to text
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("\nQuestion:")
print("What is diabetes?")

print("\nBase Model Response:")
print(response)



BASE MODEL TEST

Question:
What is diabetes?

Base Model Response:
Diabetes is a chronic metabolic disorder characterized by high blood sugar levels over an extended period. It occurs when the pancreas doesn't produce enough insulin or when the body can't effectively use the insulin it produces.

There are two main types of diabetes:

1. Type 1 Diabetes: This is also known as insulin-dependent diabetes or juvenile-onset diabetes. The immune system mistakenly attacks and destroys the beta cells in the pancreas that produce insulin. People with type 1 diabetes must take insulin injections to


In [ ]:
print(" Final Verification ")
print("✓ Base model:", MODEL_NAME)
print("✓ Tokenizer loaded:", tokenizer is not None)
print("✓ Model loaded:", model is not None)
print("✓ Device:", device)

if torch.cuda.is_available():
    print(
        "✓ GPU:",
        torch.cuda.get_device_name(0)
    )

 Final Verification 
✓ Base model: Qwen/Qwen2.5-1.5B-Instruct
✓ Tokenizer loaded: True
✓ Model loaded: True
✓ Device: cuda
✓ GPU: Tesla T4
